In [0]:
query = """
		SELECT 
            ROW_NUMBER() OVER(ORDER BY ci.customer_id) AS customer_key,
			ci.customer_id AS customer_id,
			ci.customer_key AS customer_number,
			ci.first_name AS first_name,
			ci.last_name AS last_name,
			cl.country AS country,
			ci.marital_status AS marital_status,
			CASE 
				WHEN ci.gender <> 'Unknown' THEN ci.gender
				ELSE COALESCE(ca.gender, 'Unknown')
			END AS gender,
			ca.birth_date AS birthdate,
			ci.created_date AS created_date
		FROM silver.crm_customers ci
		LEFT JOIN silver.erp_customers_info ca
		ON ci.customer_key = ca.customer_id
		LEFT JOIN silver.erp_locations cl
		ON ci.customer_key = cl.customer_id;
"""
df = spark.sql(query)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("gold.dim_customers")
)